# Supply Chain Digital Twin Simulation Pipeline

This notebook builds a one\-year supply chain digital twin from an uploaded CSV \(if present\)\. It loads and inspects the dataset, cleans it with robust defaults, synthesizes master data \(products, warehouses, suppliers\), runs a daily simulation to generate orders and shipments, and computes KPIs\. All results are saved to CSV files in the project root along with a data dictionary\.

In [1]:
# Supply Chain Digital Twin – Setup
import os
import re
import math
import json
import random
from datetime import date, datetime, timedelta
from typing import Dict, List, Tuple

import numpy as np
import pandas as pd

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

# Simulation window: last full calendar year
TODAY = pd.Timestamp.utcnow().normalize()
SIM_YEAR = (TODAY.year - 1)
SIM_START = pd.Timestamp(f"{SIM_YEAR}-01-01")
SIM_END = pd.Timestamp(f"{SIM_YEAR}-12-31")
SIM_DAYS = pd.date_range(SIM_START, SIM_END, freq='D')

# File names for outputs (SQL Server‑friendly)
OUTPUT_FILES = {
    'products': 'products.csv',
    'warehouses': 'warehouses.csv',
    'suppliers': 'suppliers.csv',
    'orders': 'orders.csv',
    'replenishment': 'replenishment.csv',
    'shipments': 'shipments.csv',
    'inventory_daily': 'inventory_daily.csv',
    'warehouse_performance': 'warehouse_performance.csv',
    'inventory_kpi': 'inventory_kpi.csv',
    'supply_chain_kpi': 'supply_chain_kpi.csv',
    'data_dictionary': 'data_dictionary.csv',
}

# Limits for manageable demo size on basic hardware
MAX_PRODUCTS = 150  # cap number of SKUs from dataset for simulation
MAX_WAREHOUSES = 3  # cap number of warehouses

# SQL Server‑friendly helpers
TRUE = 1
FALSE = 0

def to_sql_date(dt: pd.Timestamp) -> str:
    return pd.Timestamp(dt).strftime('%Y-%m-%d')

def sanitize_str(x: str) -> str:
    if pd.isna(x):
        return ''
    s = str(x)
    s = s.replace('\n', ' ').replace('\r', ' ')
    s = re.sub(r'\s+', ' ', s).strip()
    return s

pd.options.display.max_rows = 20
pd.options.display.width = 120

In [3]:
# Attempt to load the Kaggle High‑Dimensional Supply Chain Inventory dataset

def find_input_csv() -> str:
    # Prefer known filename; otherwise first CSV in project root
    candidates = [
        'supply_chain_dataset1.csv',
        'High-Dimensional Supply Chain Inventory Dataset.csv',
        'high_dim_supply_chain.csv',
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    # Fallback: first CSV in cwd
    for f in os.listdir('.'):
        if f.lower().endswith('.csv') and not f.lower().startswith('products'):
            return f
    raise FileNotFoundError('No input CSV found in project directory.')

INPUT_CSV = find_input_csv()
print(f"Using input file: {INPUT_CSV}")

raw = pd.read_csv(INPUT_CSV, low_memory=False)
print('Raw shape:', raw.shape)
print('\nRaw columns:\n', list(raw.columns))
print('\nSample rows:')
raw.head(3)

# Quick profile
profile = pd.DataFrame({
    'column': raw.columns,
    'dtype': raw.dtypes.astype(str).values,
    'non_null': raw.notna().sum().values,
    'nulls': raw.isna().sum().values,
    'unique': raw.nunique(dropna=True).values,
})
profile

Using input file: supply_chain_dataset1.csv
Raw shape: (91250, 15)

Raw columns:
 ['Date', 'SKU_ID', 'Warehouse_ID', 'Supplier_ID', 'Region', 'Units_Sold', 'Inventory_Level', 'Supplier_Lead_Time_Days', 'Reorder_Point', 'Order_Quantity', 'Unit_Cost', 'Unit_Price', 'Promotion_Flag', 'Stockout_Flag', 'Demand_Forecast']

Sample rows:


,column,dtype,non_null,nulls,unique
0,Date,object,91250,0,365
1,SKU_ID,object,91250,0,50
2,Warehouse_ID,object,91250,0,5
3,Supplier_ID,object,91250,0,10
4,Region,object,91250,0,4
5,Units_Sold,int64,91250,0,58
6,Inventory_Level,int64,91250,0,781
7,Supplier_Lead_Time_Days,int64,91250,0,13
8,Reorder_Point,int64,91250,0,142
9,Order_Quantity,int64,91250,0,301


In [5]:
# Data cleaning: duplicates, missing values, dates, dtypes, invalids

DATE_COL_HINTS = ['date', 'dt', 'timestamp', 'time']

REQUIRED_PRODUCT_KEYS = ['sku', 'product_id', 'product_code']
NAME_LIKE_COLS = ['product_name', 'item_name', 'sku_name', 'name']
CATEGORY_LIKE_COLS = ['category', 'product_category', 'segment']
WAREHOUSE_LIKE_COLS = ['warehouse', 'wh', 'warehouse_id', 'location']
SUPPLIER_LIKE_COLS = ['supplier', 'vendor', 'supplier_id']
COST_LIKE_COLS = ['unit_cost', 'cost', 'unit_purchase_price']
PRICE_LIKE_COLS = ['unit_price', 'price', 'sell_price', 'mrp']
ON_HAND_LIKE_COLS = ['on_hand', 'inventory', 'stock', 'qty_on_hand', 'current_inventory']
REORDER_LIKE_COLS = ['reorder_level', 'reorder_point', 'rop']
LEADTIME_LIKE_COLS = ['lead_time', 'lead_time_days']
WEIGHT_LIKE_COLS = ['weight', 'weight_kg']
VOLUME_LIKE_COLS = ['volume', 'cube', 'cube_m3']


def _first_existing(df: pd.DataFrame, candidates: List[str]) -> str:
    for c in candidates:
        if c in df.columns:
            return c
    # try case-insensitive match
    lower = {c.lower(): c for c in df.columns}
    for c in candidates:
        if c in lower:
            return lower[c]
    return ''


def parse_dates_inplace(df: pd.DataFrame) -> List[str]:
    parsed_cols = []
    for col in df.columns:
        if any(h in col.lower() for h in DATE_COL_HINTS):
            try:
                df[col] = pd.to_datetime(df[col], errors='coerce')
                parsed_cols.append(col)
            except Exception:
                pass
    return parsed_cols


def coerce_numeric(df: pd.DataFrame, cols: List[str]):
    for c in cols:
        if c and c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')


def clean_raw_dataframe(raw: pd.DataFrame) -> Dict[str, any]:
    df = raw.copy()

    # Remove duplicate rows
    before = len(df)
    df = df.drop_duplicates()
    after = len(df)
    print(f"Removed duplicates: {before - after}")

    # Parse dates
    parsed = parse_dates_inplace(df)
    if parsed:
        print('Parsed date columns:', parsed)

    # Trim strings
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].map(sanitize_str)

    # Normalize likely numeric columns
    numeric_likes = list(set(
        _first_existing(df, COST_LIKE_COLS) or ''
        for _ in [0]
    ))

    # Try to coerce some common numeric fields
    coerce_numeric(df, [_first_existing(df, COST_LIKE_COLS),
                        _first_existing(df, PRICE_LIKE_COLS),
                        _first_existing(df, ON_HAND_LIKE_COLS),
                        _first_existing(df, REORDER_LIKE_COLS),
                        _first_existing(df, LEADTIME_LIKE_COLS)])

    # Remove invalid records (negative costs/prices/lead times/stock)
    for like_cols, min_val in [
        ([ _first_existing(df, COST_LIKE_COLS) ], 0),
        ([ _first_existing(df, PRICE_LIKE_COLS) ], 0),
        ([ _first_existing(df, LEADTIME_LIKE_COLS) ], 0),
    ]:
        for c in like_cols:
            if c and c in df.columns:
                bad = df[c].notna() & (df[c] < min_val)
                if bad.any():
                    cnt = int(bad.sum())
                    df.loc[bad, c] = np.nan
                    print(f"Coerced {cnt} negative values to NaN in {c}")

    # Handle missing values with reasonable defaults
    # We'll impute where possible; some will be filled later during master data synthesis.
    # For strings: fill blanks with placeholder
    for c in df.select_dtypes(include=['object']).columns:
        df[c] = df[c].fillna('')

    # For numerics: leave NaN for now; we'll impute with medians or synthetic values downstream

    # Convert any remaining date-like objects to date (no time)
    for c in parsed:
        df[c] = pd.to_datetime(df[c], errors='coerce').dt.date

    return {
        'df_clean': df,
        'colmap': {
            'sku': _first_existing(df, REQUIRED_PRODUCT_KEYS),
            'name': _first_existing(df, NAME_LIKE_COLS),
            'category': _first_existing(df, CATEGORY_LIKE_COLS),
            'warehouse': _first_existing(df, WAREHOUSE_LIKE_COLS),
            'supplier': _first_existing(df, SUPPLIER_LIKE_COLS),
            'unit_cost': _first_existing(df, COST_LIKE_COLS),
            'unit_price': _first_existing(df, PRICE_LIKE_COLS),
            'on_hand': _first_existing(df, ON_HAND_LIKE_COLS),
            'reorder_level': _first_existing(df, REORDER_LIKE_COLS),
            'lead_time_days': _first_existing(df, LEADTIME_LIKE_COLS),
            'weight_kg': _first_existing(df, WEIGHT_LIKE_COLS),
            'volume_m3': _first_existing(df, VOLUME_LIKE_COLS),
        }
    }

clean_result = clean_raw_dataframe(raw)
df_clean = clean_result['df_clean']
colmap = clean_result['colmap']

print('\nColumn mapping used:')
print(json.dumps(colmap, indent=2))

df_clean.head(5)

Removed duplicates: 0
Parsed date columns: ['Date', 'Supplier_Lead_Time_Days']

Column mapping used:
{
  "sku": "",
  "name": "",
  "category": "",
  "warehouse": "Warehouse_ID",
  "supplier": "Supplier_ID",
  "unit_cost": "Unit_Cost",
  "unit_price": "Unit_Price",
  "on_hand": "",
  "reorder_level": "Reorder_Point",
  "lead_time_days": "",
  "weight_kg": "",
  "volume_m3": ""
}


,Date,SKU_ID,Warehouse_ID,Supplier_ID,Region,Units_Sold,Inventory_Level,Supplier_Lead_Time_Days,Reorder_Point,Order_Quantity,Unit_Cost,Unit_Price,Promotion_Flag,Stockout_Flag,Demand_Forecast
0,2024-01-01,SKU_1,WH_1,SUP_8,West,10,592,1970-01-01,379,0,13.95,20.48,0,0,8.52
1,2024-01-02,SKU_1,WH_1,SUP_8,West,17,575,1970-01-01,379,0,13.95,20.48,0,0,18.63
2,2024-01-03,SKU_1,WH_1,SUP_8,North,35,540,1970-01-01,379,0,13.95,20.48,1,0,39.62
3,2024-01-04,SKU_1,WH_1,SUP_8,South,24,516,1970-01-01,379,0,13.95,20.48,0,0,19.43
4,2024-01-05,SKU_1,WH_1,SUP_8,West,21,495,1970-01-01,379,0,13.95,20.48,0,0,18.70


In [15]:
# Build master tables with robust defaults when columns are missing

from collections import defaultdict


def synthesize_suppliers(n_suppliers: int = 5) -> pd.DataFrame:
    names = [f"Supplier {i+1}" for i in range(n_suppliers)]
    countries = ['US', 'CN', 'DE', 'IN', 'MX', 'VN', 'PL']
    rows = []
    for i, nm in enumerate(names, start=1):
        lead_mean = int(np.random.uniform(5, 14))
        lead_std = max(1.0, np.random.uniform(1, 4))
        on_time = float(np.clip(np.random.normal(0.92, 0.05), 0.7, 0.99))
        min_order = int(np.random.choice([10, 20, 50, 100]))
        order_cost = float(np.random.choice([25, 40, 60, 80]))
        rows.append({
            'supplier_id': i,
            'supplier_name': nm,
            'country': np.random.choice(countries),
            'on_time_prob': on_time,
            'lead_time_mean_days': lead_mean,
            'lead_time_std_days': lead_std,
            'min_order_qty': min_order,
            'order_cost': order_cost,
            'disruption_prob': float(np.round(np.random.uniform(0.02, 0.08), 3)),
        })
    return pd.DataFrame(rows)


def build_masters(df: pd.DataFrame, colmap: Dict[str, str]) -> Tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    # Warehouses
    if colmap['warehouse']:
        wh_names = (
            df[colmap['warehouse']]
            .replace('', np.nan)
            .dropna()
            .astype(str)
            .str.slice(0, 60)
            .value_counts()
            .index.tolist()
        )
    else:
        wh_names = []
    if not wh_names:
        wh_names = [f"WH-{i+1}" for i in range(min(MAX_WAREHOUSES, 3))]
    wh_names = wh_names[:MAX_WAREHOUSES]

    warehouses = []
    for i, nm in enumerate(wh_names, start=1):
        cap = int(np.random.choice([50_000, 75_000, 100_000, 150_000]))
        warehouses.append({
            'warehouse_id': i,
            'warehouse_name': sanitize_str(nm),
            'location': np.random.choice(['East', 'Central', 'West', 'South', 'North']),
            'capacity_units': cap,
        })
    warehouses = pd.DataFrame(warehouses)

    # Suppliers
    if colmap['supplier'] and df[colmap['supplier']].replace('', np.nan).dropna().nunique() >= 2:
        sup_names = df[colmap['supplier']].replace('', np.nan).dropna().astype(str).unique().tolist()
        sup_names = sup_names[:10]
        suppliers = synthesize_suppliers(len(sup_names))
        suppliers['supplier_name'] = [sanitize_str(x) for x in sup_names[: len(suppliers)]]
    else:
        suppliers = synthesize_suppliers(5)

    # Products
    # Determine a base product identifier
    sku_col = colmap['sku'] or (colmap['name'] if colmap['name'] else None)
    if not sku_col or sku_col not in df.columns:
        # fabricate SKUs
        df = df.copy()
        df['__synthetic_sku__'] = [f"SKU{100000+i}" for i in range(len(df))]
        sku_col = '__synthetic_sku__'

    # Deduplicate product rows by SKU/name
    prod_cols = [c for c in [sku_col, colmap['name'], colmap['category'], colmap['unit_cost'], colmap['unit_price']] if c]
    prods_raw = (
        df[prod_cols]
        .drop_duplicates()
        .head(MAX_PRODUCTS)
        .copy()
    )

    # Impute names/categories
    if colmap['name'] and colmap['name'] in prods_raw.columns:
        prods_raw['product_name'] = prods_raw[colmap['name']].replace('', np.nan).fillna('Unnamed Product')
    else:
        prods_raw['product_name'] = prods_raw[sku_col].astype(str)

    if colmap['category'] and colmap['category'] in prods_raw.columns:
        prods_raw['category'] = prods_raw[colmap['category']].replace('', np.nan).fillna('General')
    else:
        cats = ['A', 'B', 'C', 'D']
        prods_raw['category'] = np.random.choice(cats, size=len(prods_raw))

    # Costs and prices
    if colmap['unit_cost'] and colmap['unit_cost'] in prods_raw.columns:
        prods_raw['unit_cost'] = pd.to_numeric(prods_raw[colmap['unit_cost']], errors='coerce')
    else:
        prods_raw['unit_cost'] = np.random.lognormal(mean=2.5, sigma=0.5, size=len(prods_raw)).round(2)

    if colmap['unit_price'] and colmap['unit_price'] in prods_raw.columns:
        prods_raw['unit_price'] = pd.to_numeric(prods_raw[colmap['unit_price']], errors='coerce')
    else:
        prods_raw['unit_price'] = (prods_raw['unit_cost'] * np.random.uniform(1.15, 1.4, size=len(prods_raw))).round(2)

    # Ensure price >= cost
    mask = (prods_raw['unit_price'] < prods_raw['unit_cost'])
    prods_raw.loc[mask, 'unit_price'] = (prods_raw.loc[mask, 'unit_cost'] * 1.2).round(2)

    # Assign suppliers to products
    prods_raw = prods_raw.reset_index(drop=True)
    prods_raw['supplier_id'] = np.random.choice(suppliers['supplier_id'], size=len(prods_raw))

    # Safety stock and lead times
    if colmap['lead_time_days'] and colmap['lead_time_days'] in df.columns:
        lt_series = pd.to_numeric(df[colmap['lead_time_days']], errors='coerce').dropna()
        lt_default = int(np.clip(lt_series.median() if not lt_series.empty else 7, 2, 21))
    else:
        lt_default = 7

    prods_raw['lead_time_days'] = lt_default

    # Demand baseline (units/day) – infer if any historical demand columns present, else synthesize
    # Try to detect a demand/sales column
    demand_like_cols = [c for c in df.columns if re.search(r'(demand|sales|qty_sold|orders)', c, re.IGNORECASE)]
    if demand_like_cols:
        dm = pd.to_numeric(df[demand_like_cols[0]], errors='coerce')
        mu = float(np.nanmedian(dm)) if not dm.dropna().empty else 5.0
    else:
        mu = 5.0

    # Use cost/price to scale baseline demand (cheaper -> higher demand)
    cost = prods_raw['unit_cost'].fillna(mu)
    price = prods_raw['unit_price'].fillna(cost * 1.2)
    affordability = (price.rank(ascending=False) / len(price))  # 0..1, high for expensive
    base_demand = np.clip(np.round((1.0 - affordability) * 12 + np.random.normal(0, 2, len(prods_raw))), 1, None)
    prods_raw['base_daily_demand'] = base_demand.astype(int)

    # Safety stock heuristic: 1.65 * sigma_d * sqrt(LT); approximate sigma as 0.5*mean
    sigma_d = np.maximum(1.0, prods_raw['base_daily_demand'] * 0.5)
    prods_raw['safety_stock'] = (1.65 * sigma_d * np.sqrt(prods_raw['lead_time_days'])).round().astype(int)

    # Reorder level: mean_demand * LT + safety
    prods_raw['reorder_point'] = (prods_raw['base_daily_demand'] * prods_raw['lead_time_days'] + prods_raw['safety_stock']).round().astype(int)

    # Pack/weight/volume
    prods_raw['pack_size'] = np.random.choice([1, 5, 10, 20], size=len(prods_raw))
    prods_raw['weight_kg'] = np.round(np.random.uniform(0.1, 10.0, size=len(prods_raw)), 3)

    # Final product table
    products = pd.DataFrame({
        'product_id': np.arange(1, len(prods_raw) + 1, dtype=int),
        'sku': prods_raw[sku_col].astype(str).str.slice(0, 64).values,
        'product_name': prods_raw['product_name'].astype(str).str.slice(0, 255).values,
        'category': prods_raw['category'].astype(str).str.slice(0, 64).values,
        'supplier_id': prods_raw['supplier_id'].values,
        'unit_cost': prods_raw['unit_cost'].astype(float).round(2).values,
        'unit_price': prods_raw['unit_price'].astype(float).round(2).values,
        'lead_time_days': prods_raw['lead_time_days'].astype(int).values,
        'base_daily_demand': prods_raw['base_daily_demand'].astype(int).values,
        'safety_stock': prods_raw['safety_stock'].astype(int).values,
        'reorder_point': prods_raw['reorder_point'].astype(int).values,
        'pack_size': prods_raw['pack_size'].astype(int).values,
        'weight_kg': prods_raw['weight_kg'].astype(float).round(3).values,
    })

    # Limit warehouses to MAX_WAREHOUSES unique, otherwise synthesize
    return products, warehouses, suppliers

products, warehouses, suppliers = build_masters(df_clean, colmap)
print(products.shape, warehouses.shape, suppliers.shape)
products.head(5)
warehouses.head(5)
suppliers.head(5)

(150, 13) (3, 4) (10, 9)


,supplier_id,supplier_name,country,on_time_prob,lead_time_mean_days,lead_time_std_days,min_order_qty,order_cost,disruption_prob
0,1,SUP_8,VN,0.965954,6,3.452349,20,40.0,0.028
1,2,SUP_5,DE,0.905486,6,3.456205,50,25.0,0.030
2,3,SUP_9,US,0.922527,11,2.038299,10,40.0,0.049
3,4,SUP_2,VN,0.928525,7,2.819113,10,40.0,0.070
4,5,SUP_7,MX,0.945514,7,3.858393,20,80.0,0.022


In [19]:
# Demand generation with seasonality and promotions

# Python <=3.10 compatibility: define dataclass manually if needed
try:
    from dataclasses import dataclass
except Exception:
    def dataclass(cls):
        return cls

@dataclass
class DemandParams:
    seasonality_amplitude: float = 0.25   # +/- 25% annual seasonality
    weekly_pattern: List[float] = (0.9, 0.95, 1.0, 1.05, 1.1, 1.2, 0.85)  # Mon..Sun multipliers
    promo_prob_per_product: float = 0.12  # fraction of products having a promo in the year
    promo_spike_factor: float = 1.8       # demand multiplier during promo
    promo_length_days: Tuple[int, int] = (7, 21)


class DemandModel:
    def __init__(self, products: pd.DataFrame, params: DemandParams):
        self.products = products.copy()
        self.params = params
        self._build_promotions()

    def _build_promotions(self):
        n = len(self.products)
        promo_mask = np.random.rand(n) < self.params.promo_prob_per_product
        promo_start_offsets = np.random.randint(0, len(SIM_DAYS) - 30, size=n)
        promo_lengths = np.random.randint(self.params.promo_length_days[0], self.params.promo_length_days[1] + 1, size=n)
        start_dates = SIM_START + pd.to_timedelta(promo_start_offsets, unit='D')
        end_dates = start_dates + pd.to_timedelta(promo_lengths, unit='D')
        promo_start = pd.Series(pd.to_datetime(start_dates))
        promo_end = pd.Series(pd.to_datetime(end_dates))
        promo_start[~promo_mask] = pd.NaT
        promo_end[~promo_mask] = pd.NaT
        self.products['promo_start'] = promo_start.values
        self.products['promo_end'] = promo_end.values

    def daily_multiplier(self, dt: pd.Timestamp) -> float:
        # Annual seasonality: sine wave over day-of-year
        day_of_year = dt.timetuple().tm_yday
        seasonal = 1.0 + self.params.seasonality_amplitude * math.sin(2 * math.pi * (day_of_year / 365.0))
        # Weekly pattern
        w = dt.weekday()  # 0=Mon
        weekly = self.params.weekly_pattern[w]
        return seasonal * weekly

    def promo_multiplier(self, pid: int, dt: pd.Timestamp) -> float:
        row = self.products.loc[self.products['product_id'] == pid]
        if row.empty:
            return 1.0
        start = row['promo_start'].values[0]
        end = row['promo_end'].values[0]
        if pd.isna(start) or pd.isna(end):
            return 1.0
        if start <= dt <= end:
            return self.params.promo_spike_factor
        return 1.0

    def sample_demand(self, pid: int, dt: pd.Timestamp) -> int:
        row = self.products.loc[self.products['product_id'] == pid].iloc[0]
        base = row['base_daily_demand']
        mu = max(0.5, base * self.daily_multiplier(dt) * self.promo_multiplier(pid, dt))
        # Poisson draw for integer demand
        return int(np.random.poisson(mu))

# Instantiate demand model
params = DemandParams()
demand_model = DemandModel(products, params)

In [21]:
# Core discrete‑time simulation over daily ticks

from dataclasses import dataclass

@dataclass
class Policy:
    safety_stock_multiplier: float = 1.0
    cover_days_target: int = 14               # order‑up‑to will try to cover this many days of demand

@dataclass
class State:
    on_hand: int
    on_order: int


def draw_lead_time(supplier_row: pd.Series) -> int:
    mean_lt = int(supplier_row['lead_time_mean_days'])
    std_lt = float(supplier_row['lead_time_std_days'])
    disruption_prob = float(supplier_row.get('disruption_prob', 0.05))
    # Base: normal clipped at 1
    lt = int(max(1, np.random.normal(mean_lt, std_lt)))
    # Random disruptions add extra days (3‑10) and/or partials handled later
    if np.random.rand() < disruption_prob:
        lt += int(np.random.randint(3, 11))
    return lt


def place_order(order_id_seq: int, dt: pd.Timestamp, wh_id: int, prod_row: pd.Series, suppliers: pd.DataFrame, policy: Policy, state: State):
    supplier_row = suppliers.loc[suppliers['supplier_id'] == prod_row['supplier_id']].iloc[0]

    mean_demand = max(1, int(prod_row['base_daily_demand']))
    ss = int(max(0, round(prod_row['safety_stock'] * policy.safety_stock_multiplier)))
    reorder_point = int(mean_demand * prod_row['lead_time_days'] + ss)

    cover_days = policy.cover_days_target
    target_level = mean_demand * cover_days
    target_qty = max(0, int(target_level - (state.on_hand + state.on_order)))

    # Enforce minimum order quantity
    target_qty = int(max(target_qty, int(supplier_row['min_order_qty'])))
    if target_qty == 0:
        return None, None, None, order_id_seq

    promised_lt = int(supplier_row['lead_time_mean_days'])
    promised_date = dt + pd.to_timedelta(promised_lt, unit='D')
    actual_lt = draw_lead_time(supplier_row)
    delivery_date = dt + pd.to_timedelta(actual_lt, unit='D')

    # Potential partial delivery if disruption: reduce quantity 0‑20%
    delivered_qty = int(round(target_qty * np.random.uniform(0.8, 1.0)))

    order_row = {
        'order_id': order_id_seq,
        'order_date': to_sql_date(dt),
        'warehouse_id': wh_id,
        'supplier_id': int(prod_row['supplier_id']),
        'product_id': int(prod_row['product_id']),
        'order_qty': int(target_qty),
        'promised_date': to_sql_date(promised_date),
        'scenario': 'base',
    }

    repl_row = {
        'repl_id': order_id_seq,  # 1‑to‑1 with order for simplicity
        'order_id': order_id_seq,
        'trigger_date': to_sql_date(dt),
        'trigger_reason': 'ROP',
        'reorder_point_at_trigger': int(reorder_point),
        'on_hand_at_trigger': int(state.on_hand),
        'on_order_at_trigger': int(state.on_order),
        'target_qty': int(target_qty),
        'scenario': 'base',
    }

    shipment_row = {
        'shipment_id': order_id_seq,
        'order_id': order_id_seq,
        'ship_date': to_sql_date(dt + pd.to_timedelta(1, unit='D')),
        'promised_date': to_sql_date(promised_date),
        'delivery_date': to_sql_date(delivery_date),
        'delivered_qty': int(delivered_qty),
        'is_on_time': int(pd.Timestamp(delivery_date) <= pd.Timestamp(promised_date)),
        'is_in_full': int(delivered_qty >= target_qty),
    }

    order_id_seq += 1
    return order_row, repl_row, shipment_row, order_id_seq


def run_simulation(products: pd.DataFrame, warehouses: pd.DataFrame, suppliers: pd.DataFrame, demand_model: DemandModel, policy: Policy):
    # Initialize states per (warehouse, product)
    states: Dict[Tuple[int, int], State] = {}
    # Initial on hand: between ROP and 2x ROP
    for wh_id in warehouses['warehouse_id']:
        for _, pr in products.iterrows():
            rop = int(pr['reorder_point'])
            init = int(np.random.randint(max(1, rop), max(rop+1, rop * 2)))
            states[(int(wh_id), int(pr['product_id']))] = State(on_hand=init, on_order=0)

    # Collections
    inv_daily_rows = []
    order_rows, repl_rows, ship_rows = [], [], []
    order_id_seq = 1

    # Precompute warehouse capacity usage per unit (assume 1 unit = 1 slot)
    wh_capacity = {int(r['warehouse_id']): int(r['capacity_units']) for _, r in warehouses.iterrows()}

    for dt in SIM_DAYS:
        # Apply scheduled receipts arriving today
        today_receipts = [s for s in ship_rows if pd.Timestamp(s['delivery_date']) == dt]
        for s in today_receipts:
            # Find state and add delivered qty
            # Need order to get warehouse_id and product_id
            o = next(o for o in order_rows if o['order_id'] == s['order_id'])
            key = (int(o['warehouse_id']), int(o['product_id']))
            st = states[key]
            st.on_hand += int(s['delivered_qty'])
            st.on_order = max(0, st.on_order - int(s['delivered_qty']))

        # Daily flows per warehouse/product
        wh_demand_totals = {int(wid): {'demand': 0, 'sales': 0} for wid in wh_capacity.keys()}

        for wh_id in warehouses['warehouse_id']:
            for _, pr in products.iterrows():
                pid = int(pr['product_id'])
                key = (int(wh_id), pid)
                st = states[key]

                opening = st.on_hand

                # Demand and sales
                demand = demand_model.sample_demand(pid, dt)
                sales = min(opening, demand)
                lost = max(0, demand - sales)
                st.on_hand = opening - sales

                # Reorder check
                ss = int(pr['safety_stock'])
                mean_demand = int(max(1, pr['base_daily_demand']))
                reorder_point = int(mean_demand * pr['lead_time_days'] + ss * policy.safety_stock_multiplier)

                if (st.on_hand + st.on_order) <= reorder_point:
                    order_row, repl_row, shipment_row, order_id_seq = place_order(order_id_seq, dt, int(wh_id), pr, suppliers, policy, st)
                    if order_row is not None:
                        order_rows.append(order_row)
                        repl_rows.append(repl_row)
                        ship_rows.append(shipment_row)
                        st.on_order += int(order_row['order_qty'])

                closing = st.on_hand

                inv_daily_rows.append({
                    'inv_daily_id': len(inv_daily_rows) + 1,
                    'date': to_sql_date(dt),
                    'warehouse_id': int(wh_id),
                    'product_id': pid,
                    'opening_on_hand': int(opening),
                    'receipts': int(0),  # receipts applied at open; for accounting, compute as max(0, closing + sales - opening)
                    'sales_units': int(sales),
                    'demand_units': int(demand),
                    'lost_sales_units': int(lost),
                    'closing_on_hand': int(closing),
                    'on_order_eod': int(st.on_order),
                })

                wh_demand_totals[int(wh_id)]['demand'] += demand
                wh_demand_totals[int(wh_id)]['sales'] += sales

        # Warehouse performance metrics computed later from inv_daily
        pass

    # Build dataframes
    inventory_daily = pd.DataFrame(inv_daily_rows)
    orders = pd.DataFrame(order_rows) if order_rows else pd.DataFrame(columns=['order_id','order_date','warehouse_id','supplier_id','product_id','order_qty','promised_date','scenario'])
    replenishment = pd.DataFrame(repl_rows) if repl_rows else pd.DataFrame(columns=['repl_id','order_id','trigger_date','trigger_reason','reorder_point_at_trigger','on_hand_at_trigger','on_order_at_trigger','target_qty','scenario'])
    shipments = pd.DataFrame(ship_rows) if ship_rows else pd.DataFrame(columns=['shipment_id','order_id','ship_date','promised_date','delivery_date','delivered_qty','is_on_time','is_in_full'])

    return inventory_daily, orders, replenishment, shipments

policy = Policy()
inv_daily, orders, replenishment, shipments = run_simulation(products, warehouses, suppliers, demand_model, policy)
print(inv_daily.shape, orders.shape, shipments.shape)
inv_daily.head(5)
orders.head(5)
shipments.head(5)

(164250, 11) (7114, 8) (7114, 8)


,shipment_id,order_id,ship_date,promised_date,delivery_date,delivered_qty,is_on_time,is_in_full
0,1,1,2025-01-02,2025-01-08,2025-01-12,17,0,0
1,2,2,2025-01-02,2025-01-12,2025-01-15,43,0,1
2,3,3,2025-01-02,2025-01-06,2025-01-02,36,1,1
3,4,4,2025-01-02,2025-01-07,2025-01-07,29,1,0
4,5,5,2025-01-02,2025-01-06,2025-01-05,18,1,0


In [23]:
# Compute warehouse performance and KPI tables

# Join prices/costs
prod_price = products.set_index('product_id')[['unit_cost','unit_price']]

# Compute COGS and revenue per daily row
inv = inv_daily.copy()
inv = inv.join(prod_price, on='product_id')
inv['revenue'] = (inv['sales_units'] * inv['unit_price']).round(2)
inv['cogs'] = (inv['sales_units'] * inv['unit_cost']).round(2)

# Holding cost: 20% annual of unit_cost -> per day rate
ANNUAL_HOLDING_RATE = 0.20
DAILY_HOLDING_RATE = ANNUAL_HOLDING_RATE / 365.0
inv['avg_inventory_units'] = (inv['opening_on_hand'] + inv['closing_on_hand']) / 2.0
inv['holding_cost'] = (inv['avg_inventory_units'] * inv['unit_cost'] * DAILY_HOLDING_RATE).round(2)

# Warehouse performance daily table
wp_rows = []
for dt, grp in inv.groupby('date'):
    for wh_id, wh_grp in grp.groupby('warehouse_id'):
        demand = int(wh_grp['demand_units'].sum())
        sales = int(wh_grp['sales_units'].sum())
        util = min(1.0, wh_grp['closing_on_hand'].sum() / max(1, int(warehouses.loc[warehouses['warehouse_id']==wh_id, 'capacity_units'].iloc[0])))
        orders_today = int((orders['order_date'] == dt).sum()) if not orders.empty else 0
        rec_shipments = shipments.merge(orders[['order_id','warehouse_id']], on='order_id', how='left') if not shipments.empty else pd.DataFrame()
        rec_today = rec_shipments[(rec_shipments['delivery_date'] == dt) & (rec_shipments['warehouse_id'] == wh_id)] if not rec_shipments.empty else pd.DataFrame()
        on_time_ratio = float((rec_today['is_on_time'].sum() / len(rec_today))) if len(rec_today) > 0 else 1.0
        wp_rows.append({
            'wp_id': len(wp_rows)+1,
            'date': dt,
            'warehouse_id': int(wh_id),
            'utilization_pct': round(util*100, 2),
            'daily_fill_rate': round((sales/demand) if demand>0 else 1.0, 4),
            'stockout_events': int((wh_grp['lost_sales_units']>0).sum()),
            'orders_placed': int(orders_today),
            'receipts': int(len(rec_today)),
            'shipments_on_time_ratio': round(on_time_ratio, 4),
        })

warehouse_performance = pd.DataFrame(wp_rows)

# Inventory KPI per product/warehouse for the year
agg = inv.groupby(['product_id','warehouse_id']).agg(
    year=('date', lambda s: int(str(s.iloc[0])[:4])),
    avg_on_hand=('avg_inventory_units','mean'),
    cogs=('cogs','sum'),
    revenue=('revenue','sum'),
    lost_sales_units=('lost_sales_units','sum'),
    demand_units=('demand_units','sum'),
    sales_units=('sales_units','sum'),
    holding_cost=('holding_cost','sum'),
)
agg['fill_rate'] = (agg['sales_units'] / agg['demand_units']).replace([np.inf, np.nan], 1.0)
# Approximate turnover using average unit cost baseline
avg_unit_cost = float(prod_price['unit_cost'].mean()) if not prod_price.empty else 1.0
agg['turnover'] = (agg['cogs'] / (agg['avg_on_hand'] * max(1.0, avg_unit_cost))).replace([np.inf, np.nan], 0.0)
agg['profit'] = agg['revenue'] - agg['cogs'] - agg['holding_cost']
agg = agg.reset_index()
agg['kpi_id'] = np.arange(1, len(agg)+1)

inventory_kpi = agg[['kpi_id','product_id','warehouse_id','year','avg_on_hand','cogs','turnover','fill_rate','lost_sales_units','revenue','profit','holding_cost']]

# Orders KPIs for ordering cost and cycle time
if not orders.empty:
    orders_ext = orders.merge(shipments[['order_id','delivery_date']], on='order_id', how='left')
    orders_ext['order_cycle_time_days'] = (
        pd.to_datetime(orders_ext['delivery_date']) - pd.to_datetime(orders_ext['order_date'])
    ).dt.days
    # Ordering cost by supplier
    order_cost_map = suppliers.set_index('supplier_id')['order_cost']
    orders_ext['ordering_cost'] = orders_ext['supplier_id'].map(order_cost_map).fillna(50).astype(float)
    total_ordering_cost = float(orders_ext['ordering_cost'].sum())
    avg_cycle_time = float(orders_ext['order_cycle_time_days'].mean()) if len(orders_ext)>0 else 0.0
else:
    total_ordering_cost = 0.0
    avg_cycle_time = 0.0

# Supply chain aggregate KPIs
supply_chain = pd.DataFrame({
    'kpi_id': [1],
    'year': [SIM_YEAR],
    'total_revenue': [float(inv['revenue'].sum())],
    'total_profit': [float((inv['revenue'] - inv['cogs'] - inv['holding_cost']).sum() - total_ordering_cost)],
    'overall_fill_rate': [float(inv['sales_units'].sum() / max(1, inv['demand_units'].sum()))],
    'overall_otif': [float(shipments['is_on_time'].mean()) if not shipments.empty else 1.0],
    'inventory_turnover': [float((inv['cogs'].sum()) / max(1.0, (inv['avg_inventory_units'].mean() * max(1.0, avg_unit_cost))))],
    'holding_cost': [float(inv['holding_cost'].sum())],
    'ordering_cost': [float(total_ordering_cost)],
    'order_cycle_time_days': [float(avg_cycle_time)],
})

print('warehouse_performance:', warehouse_performance.shape)
print('inventory_kpi:', inventory_kpi.shape)
print('supply_chain_kpi:', supply_chain.shape)
warehouse_performance.head(3)
inventory_kpi.head(3)
supply_chain

warehouse_performance: (1095, 9)
inventory_kpi: (450, 12)
supply_chain_kpi: (1, 10)


,kpi_id,year,total_revenue,total_profit,overall_fill_rate,overall_otif,inventory_turnover,holding_cost,ordering_cost,order_cycle_time_days
0,1,2025,5629296.64,1393469.47,0.273835,0.611049,39099.068663,8823.57,392595.0,6.957689


In [25]:
# Ensure SQL Server‑friendly formatting and save CSVs

def save_csv(df: pd.DataFrame, path: str):
    # Standardize boolean to 0/1 and dates to YYYY-MM-DD
    df2 = df.copy()
    for c in df2.columns:
        if df2[c].dtype == 'bool':
            df2[c] = df2[c].astype(int)
        if 'date' in c and df2[c].dtype != 'O':
            try:
                df2[c] = pd.to_datetime(df2[c], errors='coerce').dt.strftime('%Y-%m-%d')
            except Exception:
                pass
        if df2[c].dtype == 'float64':
            df2[c] = df2[c].round(4)
    df2.to_csv(path, index=False)

# Prepare final tables with PK/FK sanity
products_tbl = products.copy()
warehouses_tbl = warehouses.copy()
suppliers_tbl = suppliers.copy()
orders_tbl = orders.copy()
replenishment_tbl = replenishment.copy()
shipments_tbl = shipments.copy()
inventory_daily_tbl = inv_daily.copy()
warehouse_performance_tbl = warehouse_performance.copy()
inventory_kpi_tbl = inventory_kpi.copy()
supply_chain_kpi_tbl = supply_chain.copy()

# Convert date columns to string for CSV
for t in [orders_tbl, replenishment_tbl, shipments_tbl, inventory_daily_tbl, warehouse_performance_tbl, supply_chain_kpi_tbl]:
    for c in [col for col in t.columns if 'date' in col]:
        t[c] = t[c].astype(str)

# Save
save_csv(products_tbl, OUTPUT_FILES['products'])
save_csv(warehouses_tbl, OUTPUT_FILES['warehouses'])
save_csv(suppliers_tbl, OUTPUT_FILES['suppliers'])
save_csv(orders_tbl, OUTPUT_FILES['orders'])
save_csv(replenishment_tbl, OUTPUT_FILES['replenishment'])
save_csv(shipments_tbl, OUTPUT_FILES['shipments'])
save_csv(inventory_daily_tbl, OUTPUT_FILES['inventory_daily'])
save_csv(warehouse_performance_tbl, OUTPUT_FILES['warehouse_performance'])
save_csv(inventory_kpi_tbl, OUTPUT_FILES['inventory_kpi'])
save_csv(supply_chain_kpi_tbl, OUTPUT_FILES['supply_chain_kpi'])

print('Saved CSV files:')
for k, p in OUTPUT_FILES.items():
    if k != 'data_dictionary':
        print(' -', p, os.path.exists(p), os.path.getsize(p) if os.path.exists(p) else 0)

# Data dictionary
schema = []

def add_schema(table_name: str, df: pd.DataFrame, pk_cols: List[str], fk_map: Dict[str, str]):
    for c in df.columns:
        schema.append({
            'table_name': table_name,
            'column_name': c,
            'data_type_example': str(df[c].dtype),
            'is_primary_key': int(c in pk_cols),
            'foreign_key_reference': fk_map.get(c, ''),
            'description': '',
        })

add_schema('products', products_tbl, ['product_id'], {'supplier_id': 'suppliers.supplier_id'})
add_schema('warehouses', warehouses_tbl, ['warehouse_id'], {})
add_schema('suppliers', suppliers_tbl, ['supplier_id'], {})
add_schema('orders', orders_tbl, ['order_id'], {'warehouse_id': 'warehouses.warehouse_id', 'supplier_id': 'suppliers.supplier_id', 'product_id': 'products.product_id'})
add_schema('replenishment', replenishment_tbl, ['repl_id'], {'order_id': 'orders.order_id'})
add_schema('shipments', shipments_tbl, ['shipment_id'], {'order_id': 'orders.order_id'})
add_schema('inventory_daily', inventory_daily_tbl, ['inv_daily_id'], {'warehouse_id': 'warehouses.warehouse_id', 'product_id': 'products.product_id'})
add_schema('warehouse_performance', warehouse_performance_tbl, ['wp_id'], {'warehouse_id': 'warehouses.warehouse_id'})
add_schema('inventory_kpi', inventory_kpi_tbl, ['kpi_id'], {'product_id': 'products.product_id', 'warehouse_id': 'warehouses.warehouse_id'})
add_schema('supply_chain_kpi', supply_chain_kpi_tbl, ['kpi_id'], {})

data_dictionary = pd.DataFrame(schema)
save_csv(data_dictionary, OUTPUT_FILES['data_dictionary'])
print('Data dictionary saved to', OUTPUT_FILES['data_dictionary'])

Saved CSV files:
 - products.csv True 8793
 - warehouses.csv True 109
 - suppliers.csv True 537
 - orders.csv True 300875
 - replenishment.csv True 293418
 - shipments.csv True 353508
 - inventory_daily.csv True 6299287
 - warehouse_performance.csv True 44466
 - inventory_kpi.csv True 32136
 - supply_chain_kpi.csv True 215
Data dictionary saved to data_dictionary.csv


In [27]:
# Basic integrity checks (PK uniqueness, FK coverage)

def assert_unique(df: pd.DataFrame, cols: List[str], name: str):
    if df.empty:
        return
    dup = df.duplicated(subset=cols).sum()
    assert dup == 0, f"Duplicate keys in {name}: {dup} rows"

assert_unique(products_tbl, ['product_id'], 'products')
assert_unique(warehouses_tbl, ['warehouse_id'], 'warehouses')
assert_unique(suppliers_tbl, ['supplier_id'], 'suppliers')
assert_unique(orders_tbl, ['order_id'], 'orders')
assert_unique(replenishment_tbl, ['repl_id'], 'replenishment')
assert_unique(shipments_tbl, ['shipment_id'], 'shipments')
assert_unique(inventory_daily_tbl, ['inv_daily_id'], 'inventory_daily')
assert_unique(warehouse_performance_tbl, ['wp_id'], 'warehouse_performance')
assert_unique(inventory_kpi_tbl, ['kpi_id'], 'inventory_kpi')
assert_unique(supply_chain_kpi_tbl, ['kpi_id'], 'supply_chain_kpi')

# FK presence checks
if not orders_tbl.empty:
    assert set(orders_tbl['product_id']).issubset(set(products_tbl['product_id']))
    assert set(orders_tbl['warehouse_id']).issubset(set(warehouses_tbl['warehouse_id']))
    assert set(orders_tbl['supplier_id']).issubset(set(suppliers_tbl['supplier_id']))

if not replenishment_tbl.empty:
    assert set(replenishment_tbl['order_id']).issubset(set(orders_tbl['order_id']))

if not shipments_tbl.empty:
    assert set(shipments_tbl['order_id']).issubset(set(orders_tbl['order_id']))

print('Integrity checks passed.')

Integrity checks passed.


In [29]:
# Run everything by re‑executing the notebook cells or simply call main‑style steps here.
# If you re-run, you will overwrite output CSVs.

print('Pipeline complete. If you just executed the cells above, CSVs are saved in the project root:')
for k, p in OUTPUT_FILES.items():
    if k != 'data_dictionary':
        print(' -', p)
print(' -', OUTPUT_FILES['data_dictionary'])

Pipeline complete. If you just executed the cells above, CSVs are saved in the project root:
 - products.csv
 - warehouses.csv
 - suppliers.csv
 - orders.csv
 - replenishment.csv
 - shipments.csv
 - inventory_daily.csv
 - warehouse_performance.csv
 - inventory_kpi.csv
 - supply_chain_kpi.csv
 - data_dictionary.csv


# Audit, Validation, and Fixes

This section audits the generated CSV outputs, repairs common data quality issues, enforces keys and referential integrity, recalculates KPIs with corrected logic, and writes back clean, SQL Server–friendly CSVs\. It also enhances the data dictionary with human\-readable descriptions and relationship notes\. Finally, it prints a concise validation report and readiness score\.

In [3]:
# Patch: refine parse_dates_inplace to avoid parsing non-date numeric columns like Supplier_Lead_Time_Days
# and add a helper for counting orders per warehouse/date used in recomputed KPIs.
import re
from typing import List
import pandas as pd

DATE_NAME_REGEX = re.compile(r"(^|[_])date($)|timestamp|datetime", re.IGNORECASE)

def parse_dates_inplace(df: pd.DataFrame) -> List[str]:
    parsed_cols = []
    for col in df.columns:
        col_lower = str(col).lower()
        # Only attempt parse for object-like columns whose name matches date-like regex
        if (df[col].dtype == 'object' or pd.api.types.is_string_dtype(df[col])) and DATE_NAME_REGEX.search(col_lower):
            try:
                # coerce invalid to NaT; keep original if parsing fails widely
                parsed = pd.to_datetime(df[col], errors='coerce')
                # Only accept if we got a decent proportion of non-NaT
                if parsed.notna().mean() > 0.5:
                    df[col] = parsed
                    parsed_cols.append(col)
            except Exception:
                pass
    return parsed_cols

# Helper to count orders for a given warehouse/date from an orders DataFrame
def orders_count_for_wh_date(orders_df: pd.DataFrame, wh_id: int, date_str: str) -> int:
    if orders_df is None or orders_df.empty:
        return 0
    # Ensure order_date is comparable as string YYYY-MM-DD
    od = orders_df.copy()
    if not pd.api.types.is_string_dtype(od['order_date']):
        od['order_date'] = pd.to_datetime(od['order_date'], errors='coerce').dt.strftime('%Y-%m-%d')
    return int(((od['warehouse_id'] == wh_id) & (od['order_date'] == date_str)).sum())

In [5]:
# Reusable audit/repair utilities
import os
import numpy as np
import pandas as pd
from typing import Dict, List, Tuple

# File map to enforce presence
REQUIRED_OUTPUTS = {
    'products': 'products.csv',
    'warehouses': 'warehouses.csv',
    'suppliers': 'suppliers.csv',
    'orders': 'orders.csv',
    'replenishment': 'replenishment.csv',
    'shipments': 'shipments.csv',
    'inventory_daily': 'inventory_daily.csv',
    'warehouse_performance': 'warehouse_performance.csv',
    'inventory_kpi': 'inventory_kpi.csv',
    'supply_chain_kpi': 'supply_chain_kpi.csv',
}

# Basic schemas (dtypes) expected for each table
SCHEMAS = {
    'products': {
        'product_id': 'int64', 'sku': 'object', 'product_name': 'object', 'category': 'object',
        'supplier_id': 'int64', 'unit_cost': 'float64', 'unit_price': 'float64', 'lead_time_days': 'int64',
        'base_daily_demand': 'int64', 'safety_stock': 'int64', 'reorder_point': 'int64', 'pack_size': 'int64', 'weight_kg': 'float64'
    },
    'warehouses': {
        'warehouse_id': 'int64', 'warehouse_name': 'object', 'location': 'object', 'capacity_units': 'int64'
    },
    'suppliers': {
        'supplier_id': 'int64', 'supplier_name': 'object', 'country': 'object', 'on_time_prob': 'float64',
        'lead_time_mean_days': 'int64', 'lead_time_std_days': 'float64', 'min_order_qty': 'int64', 'order_cost': 'float64', 'disruption_prob': 'float64'
    },
    'orders': {
        'order_id': 'int64', 'order_date': 'object', 'warehouse_id': 'int64', 'supplier_id': 'int64',
        'product_id': 'int64', 'order_qty': 'int64', 'promised_date': 'object', 'scenario': 'object'
    },
    'replenishment': {
        'repl_id': 'int64', 'order_id': 'int64', 'trigger_date': 'object', 'trigger_reason': 'object',
        'reorder_point_at_trigger': 'int64', 'on_hand_at_trigger': 'int64', 'on_order_at_trigger': 'int64',
        'target_qty': 'int64', 'scenario': 'object', 'product_id': 'int64', 'warehouse_id': 'int64'
    },
    'shipments': {
        'shipment_id': 'int64', 'order_id': 'int64', 'ship_date': 'object', 'promised_date': 'object',
        'delivery_date': 'object', 'delivered_qty': 'int64', 'is_on_time': 'int64', 'is_in_full': 'int64'
    },
    'inventory_daily': {
        'inv_daily_id': 'int64', 'date': 'object', 'warehouse_id': 'int64', 'product_id': 'int64',
        'opening_on_hand': 'int64', 'receipts': 'int64', 'sales_units': 'int64', 'demand_units': 'int64',
        'lost_sales_units': 'int64', 'closing_on_hand': 'int64', 'on_order_eod': 'int64'
    },
    'warehouse_performance': {
        'wp_id': 'int64', 'date': 'object', 'warehouse_id': 'int64', 'utilization_pct': 'float64',
        'daily_fill_rate': 'float64', 'stockout_events': 'int64', 'orders_placed': 'int64', 'receipts': 'int64', 'shipments_on_time_ratio': 'float64'
    },
    'inventory_kpi': {
        'kpi_id': 'int64', 'product_id': 'int64', 'warehouse_id': 'int64', 'year': 'int64', 'avg_on_hand': 'float64',
        'cogs': 'float64', 'turnover': 'float64', 'fill_rate': 'float64', 'lost_sales_units': 'int64', 'revenue': 'float64', 'profit': 'float64', 'holding_cost': 'float64'
    },
    'supply_chain_kpi': {
        'kpi_id': 'int64', 'year': 'int64', 'total_revenue': 'float64', 'total_profit': 'float64', 'overall_fill_rate': 'float64',
        'overall_otif': 'float64', 'inventory_turnover': 'float64', 'holding_cost': 'float64', 'ordering_cost': 'float64',
        'order_cycle_time_days': 'float64', 'avg_promised_lead_time_days': 'float64', 'avg_actual_lead_time_days': 'float64', 'avg_delivery_time_days': 'float64', 'demand_forecast_accuracy_mape': 'float64'
    }
}

PKS = {
    'products': ['product_id'],
    'warehouses': ['warehouse_id'],
    'suppliers': ['supplier_id'],
    'orders': ['order_id'],
    'replenishment': ['repl_id'],
    'shipments': ['shipment_id'],
    'inventory_daily': ['inv_daily_id'],
    'warehouse_performance': ['wp_id'],
    'inventory_kpi': ['kpi_id'],
    'supply_chain_kpi': ['kpi_id'],
}

FKS = {
    'orders': {
        'product_id': ('products', 'product_id'),
        'warehouse_id': ('warehouses', 'warehouse_id'),
        'supplier_id': ('suppliers', 'supplier_id'),
    },
    'replenishment': {
        'order_id': ('orders', 'order_id'),
    },
    'shipments': {
        'order_id': ('orders', 'order_id'),
    },
    'inventory_daily': {
        'product_id': ('products', 'product_id'),
        'warehouse_id': ('warehouses', 'warehouse_id'),
    },
    'warehouse_performance': {
        'warehouse_id': ('warehouses', 'warehouse_id'),
    },
    'inventory_kpi': {
        'product_id': ('products', 'product_id'),
        'warehouse_id': ('warehouses', 'warehouse_id'),
    }
}

# Stats collector
fix_stats = {
    'missing_files_created': [],
    'duplicates_removed': 0,
    'pk_fixes': 0,
    'fk_rows_dropped': 0,
    'type_coercions': 0,
    'negative_value_fixes': 0,
    'inventory_repairs': 0,
    'shipment_repairs': 0,
    'kpi_recomputed': False,
}


def ensure_csv_files(outputs_map: Dict[str, str]) -> None:
    # Use in-memory *_tbl if defined, else base dataframes, else skip
    global products_tbl, warehouses_tbl, suppliers_tbl, orders_tbl, replenishment_tbl, shipments_tbl
    global inventory_daily_tbl, warehouse_performance_tbl, inventory_kpi_tbl, supply_chain_kpi_tbl
    for name, path in outputs_map.items():
        if os.path.exists(path):
            continue
        # Determine candidate df
        df_candidate = None
        tbl_var = globals().get(f"{name}_tbl", None)
        if isinstance(tbl_var, pd.DataFrame):
            df_candidate = tbl_var
        else:
            # fall back to base dataframes created earlier
            base_map = {
                'products': globals().get('products'),
                'warehouses': globals().get('warehouses'),
                'suppliers': globals().get('suppliers'),
                'orders': globals().get('orders'),
                'replenishment': globals().get('replenishment'),
                'shipments': globals().get('shipments'),
                'inventory_daily': globals().get('inv_daily'),
                'warehouse_performance': globals().get('warehouse_performance'),
                'inventory_kpi': globals().get('inventory_kpi'),
                'supply_chain_kpi': globals().get('supply_chain'),
            }
            df_candidate = base_map.get(name, None)
        if isinstance(df_candidate, pd.DataFrame) and not df_candidate.empty:
            df_candidate.to_csv(path, index=False)
            fix_stats['missing_files_created'].append(path)


def load_all_tables(outputs_map: Dict[str, str]) -> Dict[str, pd.DataFrame]:
    tables = {}
    for name, path in outputs_map.items():
        if not os.path.exists(path):
            continue
        tables[name] = pd.read_csv(path, low_memory=False)
    return tables


def coerce_schema(name: str, df: pd.DataFrame) -> pd.DataFrame:
    schema = SCHEMAS[name]
    # Add missing columns with sensible defaults
    for col, dtype in schema.items():
        if col not in df.columns:
            if dtype.startswith('int'):
                df[col] = 0
            elif dtype.startswith('float'):
                df[col] = 0.0
            else:
                df[col] = ''
    # Keep only known columns plus any extras (we keep extras to avoid data loss)
    # Coerce types
    for col, dtype in schema.items():
        if col in df.columns:
            if dtype == 'object':
                df[col] = df[col].astype(str)
                fix_stats['type_coercions'] += 1
            elif dtype.startswith('int'):
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0).astype('int64')
                fix_stats['type_coercions'] += 1
            elif dtype.startswith('float'):
                df[col] = pd.to_numeric(df[col], errors='coerce').fillna(0.0).astype('float64')
                fix_stats['type_coercions'] += 1
    # Standardize date-like strings
    date_cols = [c for c in df.columns if re.search(r'(?:^|_)date$', c, re.IGNORECASE)]
    for c in date_cols:
        df[c] = pd.to_datetime(df[c], errors='coerce').dt.strftime('%Y-%m-%d')
    return df


def remove_duplicates(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df2 = df.drop_duplicates()
    removed = before - len(df2)
    fix_stats['duplicates_removed'] += removed
    return df2


def enforce_pk(name: str, df: pd.DataFrame) -> Tuple[pd.DataFrame, Dict[int, int]]:
    # Ensure PK non-null and unique; if not, reassign sequential IDs and return mapping
    pk_cols = PKS[name]
    mapping = {}
    if not pk_cols:
        return df, mapping
    pk = pk_cols[0]
    # If any nulls or duplicates
    if df[pk].isna().any() or df[pk].duplicated().any():
        new_ids = np.arange(1, len(df) + 1, dtype=int)
        old_ids = df[pk].copy()
        df[pk] = new_ids
        # Build mapping for FKs where applicable
        try:
            mapping = dict(zip(old_ids.astype('Int64').fillna(-1).astype(int), new_ids))
        except Exception:
            mapping = {}
        fix_stats['pk_fixes'] += 1
    return df, mapping


def apply_fk_mappings(child_df: pd.DataFrame, fk_col: str, mapping: Dict[int, int]) -> pd.DataFrame:
    if not mapping:
        return child_df
    if fk_col not in child_df.columns:
        return child_df
    # Map existing ids; unknowns remain as-is
    try:
        child_df[fk_col] = child_df[fk_col].map(lambda x: mapping.get(int(x), x))
    except Exception:
        pass
    return child_df


def validate_and_repair_fks(tables: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    # Apply PK mappings in topological order: parents first
    # Enforce PKs and collect mappings
    pk_mappings: Dict[Tuple[str, str], Dict[int, int]] = {}
    parent_order = ['products', 'warehouses', 'suppliers', 'orders']
    for parent in parent_order:
        if parent in tables:
            tables[parent] = remove_duplicates(coerce_schema(parent, tables[parent]))
            tables[parent], mapping = enforce_pk(parent, tables[parent])
            if mapping:
                pk_mappings[(parent, PKS[parent][0])] = mapping
    # Apply mappings to children and drop orphan rows
    for child, fks in FKS.items():
        if child not in tables:
            continue
        df = tables[child]
        df = remove_duplicates(coerce_schema(child, df))
        for fk_col, (parent, parent_pk) in fks.items():
            if fk_col not in df.columns or parent not in tables:
                continue
            # Map if parent PKs changed
            map_key = (parent, parent_pk)
            if map_key in pk_mappings:
                df = apply_fk_mappings(df, fk_col, pk_mappings[map_key])
            # Drop orphans
            parent_ids = set(tables[parent][parent_pk].astype(int))
            before = len(df)
            df = df[df[fk_col].astype(int).isin(parent_ids)]
            fix_stats['fk_rows_dropped'] += (before - len(df))
        tables[child] = df
        # Enforce child's PK at the end
        tables[child], _ = enforce_pk(child, tables[child])
    return tables


def clip_nonneg(df: pd.DataFrame, cols: List[str]) -> int:
    fixes = 0
    for c in cols:
        if c in df.columns:
            neg = df[c] < 0
            if neg.any():
                df.loc[neg, c] = 0
                fixes += int(neg.sum())
    return fixes


def fix_inventory_rows(inv: pd.DataFrame) -> pd.DataFrame:
    if inv.empty:
        return inv
    fixes = 0
    # Clip non-negative columns
    int_cols = ['opening_on_hand','receipts','sales_units','demand_units','lost_sales_units','closing_on_hand','on_order_eod']
    for c in int_cols:
        if c in inv.columns:
            inv[c] = pd.to_numeric(inv[c], errors='coerce').fillna(0).astype(int)
    fixes += clip_nonneg(inv, int_cols)
    # Ensure sales <= demand
    over_sales = inv['sales_units'] > inv['demand_units']
    if over_sales.any():
        inv.loc[over_sales, 'sales_units'] = inv.loc[over_sales, 'demand_units']
        fixes += int(over_sales.sum())
    # Recompute lost_sales and closing_on_hand
    inv['lost_sales_units'] = (inv['demand_units'] - inv['sales_units']).clip(lower=0)
    inv['closing_on_hand'] = (inv['opening_on_hand'] + inv['receipts'] - inv['sales_units']).clip(lower=0)
    fix_stats['inventory_repairs'] += fixes
    return inv


def repair_shipments(ship: pd.DataFrame, orders_df: pd.DataFrame) -> pd.DataFrame:
    if ship.empty:
        return ship
    fixes = 0
    # Coerce quantities
    for c in ['delivered_qty']:
        if c in ship.columns:
            ship[c] = pd.to_numeric(ship[c], errors='coerce').fillna(0).astype(int)
    # Ensure delivered <= order_qty
    if not orders_df.empty and 'order_qty' in orders_df.columns:
        qty_map = orders_df.set_index('order_id')['order_qty']
        ship = ship.merge(qty_map.rename('order_qty'), left_on='order_id', right_index=True, how='left')
        over = ship['delivered_qty'] > ship['order_qty']
        if over.any():
            ship.loc[over, 'delivered_qty'] = ship.loc[over, 'order_qty']
            fixes += int(over.sum())
    # Recompute is_on_time and is_in_full
    for col in ['ship_date','promised_date','delivery_date']:
        if col in ship.columns:
            ship[col] = pd.to_datetime(ship[col], errors='coerce').dt.strftime('%Y-%m-%d')
    ship['is_on_time'] = ((pd.to_datetime(ship['delivery_date']) <= pd.to_datetime(ship['promised_date'])).astype(int))
    if 'order_qty' in ship.columns:
        ship['is_in_full'] = (ship['delivered_qty'] >= ship['order_qty']).astype(int)
        ship.drop(columns=['order_qty'], inplace=True)
    fix_stats['shipment_repairs'] += fixes
    return ship


def enrich_replenishment(rep: pd.DataFrame, orders_df: pd.DataFrame) -> pd.DataFrame:
    if rep.empty:
        return rep
    cols_needed = ['product_id','warehouse_id']
    if all(c in rep.columns for c in cols_needed):
        return rep
    add = orders_df[['order_id','product_id','warehouse_id']] if not orders_df.empty else pd.DataFrame(columns=['order_id','product_id','warehouse_id'])
    rep = rep.merge(add, on='order_id', how='left')
    # Fill missing numeric with 0
    for c in cols_needed:
        if c in rep.columns:
            rep[c] = pd.to_numeric(rep[c], errors='coerce').fillna(0).astype(int)
    return rep


def recompute_kpis(tables: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    prods = tables.get('products', pd.DataFrame())
    inv = tables.get('inventory_daily', pd.DataFrame()).copy()
    orders_df = tables.get('orders', pd.DataFrame()).copy()
    shipments_df = tables.get('shipments', pd.DataFrame()).copy()
    warehouses_df = tables.get('warehouses', pd.DataFrame()).copy()
    suppliers_df = tables.get('suppliers', pd.DataFrame()).copy()

    # Prices/costs join
    if not prods.empty:
        price = prods.set_index('product_id')[['unit_cost','unit_price']]
        inv = inv.join(price, on='product_id')
    else:
        inv['unit_cost'] = inv['unit_cost'] if 'unit_cost' in inv else 1.0
        inv['unit_price'] = inv['unit_price'] if 'unit_price' in inv else 1.2

    # Revenue/COGS/Holding
    inv['revenue'] = (inv['sales_units'] * inv['unit_price']).astype(float)
    inv['cogs'] = (inv['sales_units'] * inv['unit_cost']).astype(float)
    DAILY_HOLDING_RATE = 0.20 / 365.0
    inv['avg_inventory_units'] = (inv['opening_on_hand'] + inv['closing_on_hand']) / 2.0
    inv['holding_cost'] = inv['avg_inventory_units'] * inv['unit_cost'] * DAILY_HOLDING_RATE

    # Warehouse performance per day/wh
    wp_rows = []
    # Prepare receipts: shipments joined to orders for wh
    if not shipments_df.empty and not orders_df.empty:
        rec_ship = shipments_df.merge(orders_df[['order_id','warehouse_id']], on='order_id', how='left')
    else:
        rec_ship = pd.DataFrame(columns=['delivery_date','warehouse_id','is_on_time'])
    for (dt, wh_id), grp in inv.groupby(['date','warehouse_id']):
        demand = int(grp['demand_units'].sum())
        sales = int(grp['sales_units'].sum())
        # utilization based on closing inventory vs capacity
        if not warehouses_df.empty:
            cap = int(warehouses_df.loc[warehouses_df['warehouse_id']==int(wh_id), 'capacity_units'].iloc[0]) if (warehouses_df['warehouse_id']==int(wh_id)).any() else 1
        else:
            cap = 1
        util = min(1.0, float(grp['closing_on_hand'].sum()) / max(1, cap))
        date_str = str(dt)
        orders_cnt = 0
        if not orders_df.empty:
            # Ensure order_date normalized
            od = orders_df.copy()
            od['order_date'] = pd.to_datetime(od['order_date'], errors='coerce').dt.strftime('%Y-%m-%d')
            orders_cnt = int(((od['warehouse_id'] == int(wh_id)) & (od['order_date'] == date_str)).sum())
        # receipts for that wh/date
        rec_today = rec_ship[(rec_ship['warehouse_id']==int(wh_id)) & (pd.to_datetime(rec_ship['delivery_date'], errors='coerce').dt.strftime('%Y-%m-%d')==date_str)]
        on_time_ratio = float(rec_today['is_on_time'].mean()) if len(rec_today)>0 else 1.0
        wp_rows.append({
            'date': date_str,
            'warehouse_id': int(wh_id),
            'utilization_pct': round(util*100, 2),
            'daily_fill_rate': round((sales/demand) if demand>0 else 1.0, 4),
            'stockout_events': int((grp['lost_sales_units']>0).sum()),
            'orders_placed': int(orders_cnt),
            'receipts': int(len(rec_today)),
            'shipments_on_time_ratio': round(on_time_ratio, 4),
        })
    warehouse_performance = pd.DataFrame(wp_rows)
    warehouse_performance.insert(0, 'wp_id', np.arange(1, len(warehouse_performance)+1, dtype=int))

    # Inventory KPI per product/warehouse/year
    if not inv.empty:
        inv_kpi = inv.copy()
        inv_kpi['year'] = pd.to_datetime(inv_kpi['date'], errors='coerce').dt.year
        agg = inv_kpi.groupby(['product_id','warehouse_id','year']).agg(
            avg_on_hand=('avg_inventory_units','mean'),
            cogs=('cogs','sum'),
            revenue=('revenue','sum'),
            lost_sales_units=('lost_sales_units','sum'),
            demand_units=('demand_units','sum'),
            sales_units=('sales_units','sum'),
            holding_cost=('holding_cost','sum'),
        ).reset_index()
        agg['fill_rate'] = (agg['sales_units'] / agg['demand_units']).replace([np.inf, np.nan], 1.0)
        # Use actual unit costs by product for inventory value. Approx via mean unit_cost per product
        unit_cost_map = inv_kpi.groupby('product_id')['unit_cost'].mean()
        avg_inv_value = agg['avg_on_hand'] * agg['product_id'].map(unit_cost_map).fillna(unit_cost_map.mean())
        agg['turnover'] = (agg['cogs'] / avg_inv_value).replace([np.inf, np.nan], 0.0)
        agg['profit'] = agg['revenue'] - agg['cogs'] - agg['holding_cost']
        agg['kpi_id'] = np.arange(1, len(agg)+1)
        inventory_kpi = agg[['kpi_id','product_id','warehouse_id','year','avg_on_hand','cogs','turnover','fill_rate','lost_sales_units','revenue','profit','holding_cost']]
    else:
        inventory_kpi = pd.DataFrame(columns=list(SCHEMAS['inventory_kpi'].keys()))

    # Supply chain KPI
    if not orders_df.empty and not shipments_df.empty:
        orders_ext = orders_df.merge(shipments_df[['order_id','delivery_date','is_on_time','is_in_full']], on='order_id', how='left')
        orders_ext['order_cycle_time_days'] = (pd.to_datetime(orders_ext['delivery_date']) - pd.to_datetime(orders_ext['order_date'])).dt.days
        # Ordering cost
        if not suppliers_df.empty and 'order_cost' in suppliers_df.columns:
            order_cost_map = suppliers_df.set_index('supplier_id')['order_cost']
            orders_ext['ordering_cost'] = orders_ext['supplier_id'].map(order_cost_map).fillna(order_cost_map.mean() if len(order_cost_map)>0 else 50).astype(float)
        else:
            orders_ext['ordering_cost'] = 50.0
        total_ordering_cost = float(orders_ext['ordering_cost'].sum())
        # OTIF as on_time AND in_full
        orders_ext['otif'] = ((orders_ext['is_on_time']==1) & (orders_ext['is_in_full']==1)).astype(int)
        overall_otif = float(orders_ext['otif'].mean()) if len(orders_ext)>0 else 1.0
        avg_cycle_time = float(orders_ext['order_cycle_time_days'].mean()) if len(orders_ext)>0 else 0.0
        # Lead times
        promised_lt = (pd.to_datetime(orders_ext['promised_date']) - pd.to_datetime(orders_ext['order_date'])).dt.days
        actual_lt = (pd.to_datetime(orders_ext['delivery_date']) - pd.to_datetime(orders_ext['order_date'])).dt.days
        avg_promised_lt = float(promised_lt.mean()) if len(promised_lt)>0 else 0.0
        avg_actual_lt = float(actual_lt.mean()) if len(actual_lt)>0 else 0.0
        # Delivery time from ship_date to delivery
        avg_delivery_time = float((pd.to_datetime(orders_ext['delivery_date']) - pd.to_datetime(orders_ext['ship_date'])).dt.days.mean()) if 'ship_date' in orders_ext.columns else avg_actual_lt
    else:
        total_ordering_cost = 0.0
        overall_otif = 1.0
        avg_cycle_time = 0.0
        avg_promised_lt = 0.0
        avg_actual_lt = 0.0
        avg_delivery_time = 0.0

    total_revenue = float(inv['revenue'].sum()) if not inv.empty else 0.0
    holding_cost_total = float(inv['holding_cost'].sum()) if not inv.empty else 0.0
    total_profit = float((inv['revenue'] - inv['cogs'] - inv['holding_cost']).sum() - total_ordering_cost) if not inv.empty else 0.0
    overall_fill_rate = float(inv['sales_units'].sum() / max(1, inv['demand_units'].sum())) if not inv.empty else 1.0
    avg_unit_cost = float(inv['unit_cost'].mean()) if 'unit_cost' in inv else 1.0
    inventory_turnover = float((inv['cogs'].sum()) / max(1.0, (inv['avg_inventory_units'].mean() * max(1.0, avg_unit_cost)))) if not inv.empty else 0.0

    # Forecast accuracy MAPE using base_daily_demand as naive forecast
    if not prods.empty and not inv.empty:
        base_map = prods.set_index('product_id')['base_daily_demand']
        inv['forecast'] = inv['product_id'].map(base_map).fillna(base_map.mean() if len(base_map)>0 else 0)
        # Avoid divide-by-zero: use demand>0
        valid = inv['demand_units'] > 0
        mape = float((np.abs(inv.loc[valid, 'demand_units'] - inv.loc[valid, 'forecast']) / inv.loc[valid, 'demand_units']).mean()) if valid.any() else 0.0
    else:
        mape = 0.0

    supply_chain_kpi = pd.DataFrame({
        'kpi_id': [1],
        'year': [int(pd.to_datetime(inv['date'].iloc[0]).year) if not inv.empty else SIM_YEAR],
        'total_revenue': [total_revenue],
        'total_profit': [total_profit],
        'overall_fill_rate': [overall_fill_rate],
        'overall_otif': [overall_otif],
        'inventory_turnover': [inventory_turnover],
        'holding_cost': [holding_cost_total],
        'ordering_cost': [total_ordering_cost],
        'order_cycle_time_days': [avg_cycle_time],
        'avg_promised_lead_time_days': [avg_promised_lt],
        'avg_actual_lead_time_days': [avg_actual_lt],
        'avg_delivery_time_days': [avg_delivery_time],
        'demand_forecast_accuracy_mape': [mape],
    })

    tables['warehouse_performance'] = warehouse_performance
    tables['inventory_kpi'] = inventory_kpi
    tables['supply_chain_kpi'] = supply_chain_kpi
    fix_stats['kpi_recomputed'] = True
    return tables


def build_data_dictionary_enhanced(tables: Dict[str, pd.DataFrame]) -> pd.DataFrame:
    descriptions = {
        'products': {
            'product_id': 'Surrogate key for product',
            'sku': 'Product SKU or code',
            'product_name': 'Human-readable product name',
            'category': 'Product category or segment',
            'supplier_id': 'FK to suppliers.supplier_id',
            'unit_cost': 'Per-unit cost',
            'unit_price': 'Per-unit selling price',
            'lead_time_days': 'Average supplier lead time in days',
            'base_daily_demand': 'Naive baseline daily demand used for forecasting',
            'safety_stock': 'Buffer stock level',
            'reorder_point': 'Inventory level triggering replenishment',
            'pack_size': 'Units per pack',
            'weight_kg': 'Weight per unit in kg',
        },
        'warehouses': {
            'warehouse_id': 'Surrogate key for warehouse',
            'warehouse_name': 'Warehouse name/code',
            'location': 'Region/location label',
            'capacity_units': 'Storage capacity in units',
        },
        'suppliers': {
            'supplier_id': 'Surrogate key for supplier',
            'supplier_name': 'Supplier name/code',
            'country': 'Supplier country',
            'on_time_prob': 'Estimated on-time delivery probability',
            'lead_time_mean_days': 'Mean promised lead time in days',
            'lead_time_std_days': 'Lead time variability (std dev)',
            'min_order_qty': 'Minimum order quantity',
            'order_cost': 'Per-order administrative cost',
            'disruption_prob': 'Probability of supply disruption',
        },
        'orders': {
            'order_id': 'Purchase order identifier',
            'order_date': 'Order creation date',
            'warehouse_id': 'FK to warehouses.warehouse_id',
            'supplier_id': 'FK to suppliers.supplier_id',
            'product_id': 'FK to products.product_id',
            'order_qty': 'Quantity ordered',
            'promised_date': 'Promised delivery date',
            'scenario': 'Scenario label',
        },
        'replenishment': {
            'repl_id': 'Replenishment trigger identifier',
            'order_id': 'FK to orders.order_id',
            'trigger_date': 'Date of trigger',
            'trigger_reason': 'Reason for trigger (e.g., ROP)',
            'reorder_point_at_trigger': 'ROP at time of trigger',
            'on_hand_at_trigger': 'On-hand at time of trigger',
            'on_order_at_trigger': 'On-order at time of trigger',
            'target_qty': 'Target reorder quantity',
            'scenario': 'Scenario label',
            'product_id': 'FK to products.product_id (enriched)',
            'warehouse_id': 'FK to warehouses.warehouse_id (enriched)',
        },
        'shipments': {
            'shipment_id': 'Shipment identifier',
            'order_id': 'FK to orders.order_id',
            'ship_date': 'Ship date',
            'promised_date': 'Promised delivery date',
            'delivery_date': 'Actual delivery date',
            'delivered_qty': 'Delivered quantity',
            'is_on_time': '1 if delivered by promised date, else 0',
            'is_in_full': '1 if delivered quantity >= ordered quantity, else 0',
        },
        'inventory_daily': {
            'inv_daily_id': 'Daily inventory fact identifier',
            'date': 'Calendar date',
            'warehouse_id': 'FK to warehouses.warehouse_id',
            'product_id': 'FK to products.product_id',
            'opening_on_hand': 'Units at start of day',
            'receipts': 'Units received during day',
            'sales_units': 'Units sold during day',
            'demand_units': 'Customer demand units',
            'lost_sales_units': 'Unmet demand units',
            'closing_on_hand': 'Units at end of day',
            'on_order_eod': 'Units on order at end of day',
        },
        'warehouse_performance': {
            'wp_id': 'Warehouse daily performance identifier',
            'date': 'Calendar date',
            'warehouse_id': 'FK to warehouses.warehouse_id',
            'utilization_pct': 'Utilization percentage of capacity',
            'daily_fill_rate': 'Sales/Demand per day',
            'stockout_events': 'Count of stockout occurrences',
            'orders_placed': 'Number of POs created that day at the warehouse',
            'receipts': 'Number of shipments delivered that day at the warehouse',
            'shipments_on_time_ratio': 'Share of receipts delivered on or before promised date',
        },
        'inventory_kpi': {
            'kpi_id': 'Inventory KPI per product-warehouse-year',
            'product_id': 'FK to products.product_id',
            'warehouse_id': 'FK to warehouses.warehouse_id',
            'year': 'Calendar year of KPI',
            'avg_on_hand': 'Average daily on-hand units',
            'cogs': 'Cost of goods sold for the year',
            'turnover': 'COGS divided by average inventory value',
            'fill_rate': 'Sales divided by demand',
            'lost_sales_units': 'Total unmet demand units',
            'revenue': 'Total revenue from sales',
            'profit': 'Revenue minus COGS and holding cost',
            'holding_cost': 'Carrying cost based on daily holding rate',
        },
        'supply_chain_kpi': {
            'kpi_id': 'Annual supply chain KPI summary',
            'year': 'Calendar year',
            'total_revenue': 'Total revenue across all warehouses/products',
            'total_profit': 'Total profit after holding and ordering costs',
            'overall_fill_rate': 'Overall sales/demand ratio',
            'overall_otif': 'Share of orders delivered on time and in full',
            'inventory_turnover': 'Overall inventory turnover',
            'holding_cost': 'Total inventory holding cost',
            'ordering_cost': 'Total ordering cost across POs',
            'order_cycle_time_days': 'Average days from order to delivery',
            'avg_promised_lead_time_days': 'Average promised lead time in days',
            'avg_actual_lead_time_days': 'Average actual lead time in days',
            'avg_delivery_time_days': 'Average ship-to-delivery time in days',
            'demand_forecast_accuracy_mape': 'Mean absolute percentage error vs baseline forecast',
        },
    }

    relationships = {
        'products': 'products.supplier_id -> suppliers.supplier_id',
        'warehouses': '',
        'suppliers': '',
        'orders': 'orders.(product_id, warehouse_id, supplier_id) -> products, warehouses, suppliers',
        'replenishment': 'replenishment.order_id -> orders.order_id; enriched with product_id, warehouse_id',
        'shipments': 'shipments.order_id -> orders.order_id',
        'inventory_daily': 'inventory_daily.(product_id, warehouse_id) -> products, warehouses',
        'warehouse_performance': 'warehouse_performance.warehouse_id -> warehouses.warehouse_id',
        'inventory_kpi': 'inventory_kpi.(product_id, warehouse_id) -> products, warehouses',
        'supply_chain_kpi': 'Aggregate KPIs; no FKs',
    }

    rows = []
    for tname, df in tables.items():
        if df is None or df.empty:
            continue
        descs = descriptions.get(tname, {})
        for c in df.columns:
            rows.append({
                'table_name': tname,
                'column_name': c,
                'data_type_example': str(df[c].dtype),
                'is_primary_key': int(c in PKS.get(tname, [])),
                'foreign_key_reference': '',
                'description': descs.get(c, ''),
                'relationship': relationships.get(tname, ''),
            })
    dd = pd.DataFrame(rows)
    return dd


def save_all(tables: Dict[str, pd.DataFrame]) -> None:
    for name, path in REQUIRED_OUTPUTS.items():
        if name in tables:
            df = tables[name].copy()
            # normalize booleans and date strings
            for c in df.columns:
                if df[c].dtype == 'bool':
                    df[c] = df[c].astype(int)
                if 'date' in c and df[c].dtype != 'O':
                    df[c] = pd.to_datetime(df[c], errors='coerce').dt.strftime('%Y-%m-%d')
                if df[c].dtype == 'float64':
                    df[c] = df[c].round(4)
            df.to_csv(path, index=False)

In [7]:
# Run the audit end-to-end and produce a validation report

# 1) Ensure required CSV files exist (regenerate if missing)
ensure_csv_files(REQUIRED_OUTPUTS)

# 2) Load all tables
_tables = load_all_tables(REQUIRED_OUTPUTS)

# 2a) Transactional fixes prior to FK validation
# Coerce and clean each table per schema, remove duplicates
for tname in list(_tables.keys()):
    _tables[tname] = remove_duplicates(coerce_schema(tname, _tables[tname]))

# Enrich replenishment with product_id and warehouse_id for later relationships
if 'replenishment' in _tables and 'orders' in _tables:
    _tables['replenishment'] = enrich_replenishment(_tables['replenishment'], _tables['orders'])

# Clip negatives and repair inventory and shipments logic
if 'products' in _tables:
    cols = ['unit_cost','unit_price','lead_time_days','base_daily_demand','safety_stock','reorder_point','pack_size','weight_kg']
    for c in cols:
        if c in _tables['products'].columns:
            ser = pd.to_numeric(_tables['products'][c], errors='coerce')
            neg = ser < 0
            if neg.any():
                _tables['products'].loc[neg, c] = 0
                fix_stats['negative_value_fixes'] += int(neg.sum())

if 'inventory_daily' in _tables:
    _tables['inventory_daily'] = fix_inventory_rows(_tables['inventory_daily'])

if 'shipments' in _tables:
    _tables['shipments'] = repair_shipments(_tables['shipments'], _tables.get('orders', pd.DataFrame()))

# 2b) Enforce PKs and validate/repair FKs (parents first, cascade via drops)
_tables = validate_and_repair_fks(_tables)

# 3) Recompute KPI tables from cleaned transactions
_tables = recompute_kpis(_tables)

# 4) Save corrected tables
save_all(_tables)

# 5) Build enhanced data dictionary and save
_data_dictionary = build_data_dictionary_enhanced(_tables)
_data_dictionary.to_csv('data_dictionary.csv', index=False)

# 6) Validation report and readiness score
missing_created = fix_stats.get('missing_files_created', [])
dups_removed = fix_stats.get('duplicates_removed', 0)
pk_fixes = fix_stats.get('pk_fixes', 0)
fk_dropped = fix_stats.get('fk_rows_dropped', 0)
type_coercions = fix_stats.get('type_coercions', 0)
neg_fixes = fix_stats.get('negative_value_fixes', 0)
inv_repairs = fix_stats.get('inventory_repairs', 0)
ship_repairs = fix_stats.get('shipment_repairs', 0)
kpi_recomputed = fix_stats.get('kpi_recomputed', False)

# Simple readiness score heuristic
score = 100
score -= min(20, dups_removed // 1000)
score -= min(10, pk_fixes * 2)
score -= min(10, fk_dropped // 1000)
score -= min(10, neg_fixes // 1000)
score = max(0, min(100, score))

print('Audit & Validation Report')
print('--------------------------')
print(f"Missing files created: {len(missing_created)} -> {missing_created}")
print(f"Duplicates removed: {dups_removed}")
print(f"Primary key fixes applied: {pk_fixes}")
print(f"Foreign key orphan rows dropped: {fk_dropped}")
print(f"Type coercions performed: {type_coercions}")
print(f"Negative/invalid value fixes: {neg_fixes}")
print(f"Inventory repairs: {inv_repairs}")
print(f"Shipment repairs: {ship_repairs}")
print(f"KPI recomputed: {kpi_recomputed}")
print(f"Project readiness score: {score}/100")

print('\nChecklist:')
print(f"✔ CSV presence ensured")
print(f"✔ Duplicates removed")
print(f"✔ PKs enforced and fixed if needed")
print(f"✔ FKs validated and repaired; cascades applied")
print(f"✔ Data types coerced; dates standardized; booleans normalized")
print(f"✔ Inventory rows repaired and consistent")
print(f"✔ Shipments consistent with orders and flags recomputed")
print(f"✔ KPIs recalculated from cleaned data")

# Preview several tables to confirm shape
for name in ['products','warehouses','suppliers','orders','replenishment','shipments','inventory_daily','warehouse_performance','inventory_kpi','supply_chain_kpi']:
    if name in _tables:
        print(f"\n{name} -> shape: {_tables[name].shape}")
        df = _tables[name]
        df

Audit & Validation Report
--------------------------
Missing files created: 0 -> []
Duplicates removed: 0
Primary key fixes applied: 0
Foreign key orphan rows dropped: 0
Type coercions performed: 192
Negative/invalid value fixes: 0
Inventory repairs: 0
Shipment repairs: 0
KPI recomputed: True
Project readiness score: 100/100

Checklist:
✔ CSV presence ensured
✔ Duplicates removed
✔ PKs enforced and fixed if needed
✔ FKs validated and repaired; cascades applied
✔ Data types coerced; dates standardized; booleans normalized
✔ Inventory rows repaired and consistent
✔ Shipments consistent with orders and flags recomputed
✔ KPIs recalculated from cleaned data

products -> shape: (150, 13)

warehouses -> shape: (3, 4)

suppliers -> shape: (10, 9)

orders -> shape: (7114, 8)

replenishment -> shape: (7114, 11)

shipments -> shape: (7114, 8)

inventory_daily -> shape: (164250, 11)

warehouse_performance -> shape: (1095, 9)

inventory_kpi -> shape: (450, 12)

supply_chain_kpi -> shape: (1, 14)


<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=226d03f9-fc06-4a5f-89d2-afa7c5887ba6' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>